In [1]:
# Case ficticio, empresa varejo max
# Clientes, Vendas, Vendedores, Empregados
import pandas as pd 
import os

os.makedirs('resultado', exist_ok=True)

df_clientes = pd.read_csv('clientes_varejomax_grande.csv')
df_vendas = pd.read_csv('vendas_varejomax_grande.csv')
df_vendedores = pd.read_csv('vendedores_varejomax_grande.csv')
df_empregados = pd.read_csv('empregados_varejomax_grande.csv')

# print(df_clientes.head())
# print(df_vendas.head())
# print(df_vendedores.head())
# print(df_empregados.head())

# display(df_vendas)
df_vendas['data'] = pd.to_datetime(df_vendas['data'])
df_vendas['mes'] = df_vendas['data'].dt.strftime('%m/%Y')

df_empregados['data_admissao'] = pd.to_datetime(df_empregados['data_admissao'])

df_vendas = df_vendas.merge(df_clientes, on='id_cliente')
df_vendas = df_vendas.merge(df_vendedores, on='id_vendedor')
df_vendas = df_vendas.rename(columns = {"nome_x": "nome_cliente", "nome_y": "nome_vendedor", "regiao_x": "regiao_cliente", "regiao_y": "regiao_vendedor"})

display(df_vendas.head())
df_vendas.to_excel('resultado/vendas_completa.xlsx', index=False)

melhor_vend_valor = df_vendas.groupby(['id_vendedor', 'nome_vendedor'])['valor'].sum()
melhor_vend_valor = melhor_vend_valor.sort_values(ascending=False)
melhor_vend_valor = melhor_vend_valor.reset_index()
cinco_melhores_vendedores_valor = melhor_vend_valor.head(5)
cinco_melhores_vendedores_valor.to_excel('resultado/5_melhores_vendedores.xlsx', index=False)

melhor_vend_qtde = df_vendas.groupby(['id_vendedor', 'nome_vendedor'])['id_venda'].count()
melhor_vend_qtde = melhor_vend_qtde.sort_values(ascending=False)
melhor_vend_qtde = melhor_vend_qtde.reset_index()
melhor_vend_qtde = melhor_vend_qtde.rename(columns={'id_venda': 'quantidade_vendas'})
cinco_melhor_vend_qtde = melhor_vend_qtde.head(5)
cinco_melhor_vend_qtde.to_excel('resultado/5_melhores_vendedores_qtde.xlsx', index=False)

melhor_prod_qtde = df_vendas.groupby(['produto'])['valor'].count()
melhor_prod_qtde = melhor_prod_qtde.sort_values(ascending=False)
melhor_prod_qtde = melhor_prod_qtde.reset_index()
melhor_prod_qtde = melhor_prod_qtde.rename(columns={'valor': 'quantidade_vendas'})
cinco_melhor_prod_qtde = melhor_prod_qtde.head(5)
cinco_melhor_prod_qtde.to_excel('resultado/5_melhores_produtos_qtde.xlsx', index=False)

melhor_prod_valor = df_vendas.groupby(['produto'])['valor'].sum()
melhor_prod_valor = melhor_prod_valor.sort_values(ascending=False)
melhor_prod_valor = melhor_prod_valor.reset_index()
cinco_melhor_prod_qtde = melhor_prod_valor.head(5)
cinco_melhor_prod_qtde.to_excel('resultado/5_melhores_produtos_valor.xlsx', index=False)  

vendas_regiao = df_vendas.groupby(['regiao_vendedor'])['valor'].sum()
vendas_regiao = vendas_regiao.sort_values(ascending=False)  
vendas_regiao = vendas_regiao.reset_index()
vendas_regiao.to_excel('resultado/vendas_regiao.xlsx', index=False)

faturamentototal = df_vendas['valor'].sum()

print(f'faturamento total: R${faturamentototal:.2f}')

melhor_cliente_valor = df_vendas.groupby(['id_cliente', 'nome_cliente'])['valor'].sum()
melhor_cliente_valor = melhor_cliente_valor.sort_values(ascending=False)
melhor_cliente_valor = melhor_cliente_valor.reset_index()  
cinco_melhores_cliente_valor = melhor_cliente_valor.head(5)
cinco_melhores_cliente_valor.to_excel('resultado/5_melhores_clientes.xlsx', index=False)

melhor_cliente_qtde = df_vendas.groupby(['id_cliente', 'nome_cliente'])['id_venda'].count()
melhor_cliente_qtde = melhor_cliente_qtde.sort_values(ascending=False)
melhor_cliente_qtde = melhor_cliente_qtde.reset_index()   
melhor_cliente_qtde = melhor_cliente_qtde.rename(columns={'id_venda': 'quantidade_vendas'})
cinco_melhores_cliente_qtde = melhor_cliente_qtde.head(5)
cinco_melhores_cliente_qtde.to_excel('resultado/5_melhores_clientes_qtde.xlsx', index=False)

df_vendas['mes'] = df_vendas['data'].dt.to_period('M')
vendas_mes = df_vendas.groupby('mes')['valor'].sum()
vendas_mes = vendas_mes.reset_index()
vendas_mes.to_excel('resultado/vendas_mes.xlsx', index=False)





,id_venda,data,id_cliente,id_vendedor,produto,valor,mes,nome_cliente,cidade,regiao_cliente,nome_vendedor,regiao_vendedor
0,1,2024-01-31,30,8,Mesa,943,01/2024,Cliente_30,Porto Alegre,Sul,Vendedor_8,Nordeste
1,2,2024-03-31,187,16,Notebook,3529,03/2024,Cliente_187,Rio de Janeiro,Sudeste,Vendedor_16,Nordeste
2,3,2024-03-01,76,16,Cadeira,787,03/2024,Cliente_76,Rio de Janeiro,Sudeste,Vendedor_16,Nordeste
3,4,2024-04-18,15,3,Mesa,1095,04/2024,Cliente_15,Fortaleza,Nordeste,Vendedor_3,Sudeste
4,5,2024-03-31,82,17,Monitor,1601,03/2024,Cliente_82,Porto Alegre,Sul,Vendedor_17,Sul


faturamento total: R$6344959.00


In [2]:
import plotly.express as px
import os

os.makedirs('graficos', exist_ok=True)

vendas_mes['mes'] = vendas_mes['mes'].astype(str)

df_vendas.head()

graf_vendas_mes = px.bar(
    vendas_mes,
    x='mes',
    y='valor',
    title='Faturamento por Mês', 
    labels={
            "mes": "Mês",
            "valor": "Faturamento"
    }
)
graf_vendas_mes.write_html('graficos/vendas_mes.html')
graf_vendas_mes

graf_vendas_vendedores = px.bar(
    cinco_melhores_vendedores_valor,
    x='valor',
    y='nome_vendedor',
    title='Top 5 Vendedores por Faturamento',
    labels={
        "nome_vendedor": "Vendedor",
        "valor": "Faturamento (R$)"
    },
    text_auto=True
)

graf_vendas_vendedores.update_layout(
    yaxis={'categoryorder':'total ascending'}
)

graf_vendas_vendedores.write_html('graficos/melhores_vendedores.html')

graf_vendas_vendedores
